# Lab: Materials Modelling at the Continuum Scale

**Module: Materialmodellierung | Helmut-Schmidt-Universität Hamburg**

---

In this lab you will simulate two important classes of continuum model from the lecture:

1. **The Poisson Equation** — applied to the deflection of a pre-tensioned elastic membrane under a transverse load (Lectures 3 & 4)
2. **The Cahn-Hilliard Phase Field Model** — simulating spinodal decomposition, the spontaneous unmixing of a binary alloy (Lecture 3)

Both problems are solved with the **Finite Element Method (FEM)** via the [FEniCS](https://fenicsproject.org/) library.
FEniCS takes the mathematical *weak form* of your PDE and automatically assembles the algebraic system that a computer solves.

**Learning objectives**
- Set up and solve a FEM problem in FEniCS from a PDE description
- Understand how boundary conditions and load distributions shape the Poisson solution
- Run a time-dependent phase-field simulation and observe microstructure formation
- Explain how the interface energy parameter $\kappa$ controls the length scale of phase separation

## Getting Started

Work through the cells **from top to bottom**.
Each section starts with the **physics** of the problem and is followed by the FEniCS **implementation**.

- Run a cell with **Shift + Enter**
- Cells marked `# ACTION` contain parameters you should **change and experiment with**
- Add your own notes by inserting markdown cells (*Insert → Insert Cell Below*, then change type to Markdown)
- FEniCS may print some initialisation lines the first time — this is normal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# FEniCS — all FEM functionality lives in dolfin
from dolfin import *

# Must come AFTER 'from dolfin import *': dolfin exports its own module called
# 'timer' (dolfin.common.timer) which would otherwise shadow this import.
import time as timer

# Suppress FEniCS progress messages (change to 20 for INFO, 10 for DEBUG)
set_log_level(30)

print("NumPy  version:", np.__version__)

---
## Part 1: The Poisson Equation — Elastic Membrane

### Physical Problem

Consider a thin elastic membrane stretched across a rigid square frame.
The membrane is pre-tensioned (like a drum skin) and subjected to a transverse load $f(x,y)$
(force per unit area, e.g. due to a pressure difference).
At equilibrium the deflection $u(x,y)$ from the rest position satisfies the **Poisson equation**:

$$
-\nabla^2 u(x,y) = g(x,y) \quad \text{in } \Omega = [0,1]^2
$$

where $g = f/T$ is the load normalised by the membrane tension $T$, and
$\nabla^2 = \partial^2/\partial x^2 + \partial^2/\partial y^2$ is the Laplace operator.

The membrane is clamped at all four edges, giving **Dirichlet boundary conditions**:

$$
u = 0 \quad \text{on } \partial\Omega
$$

> **Connection to the lecture:** the same equation governs heat conduction, diffusion, electrical
> potential, and incompressible pressure-driven flow — only the physical interpretation of
> $u$ and $g$ changes.

### Variational (Weak) Form

FEniCS works with the **weak form** of the PDE.
Multiplying by a test function $v$ (with $v = 0$ on $\partial\Omega$) and integrating by parts:

$$
\underbrace{\int_\Omega \nabla u \cdot \nabla v \, d\mathbf{r}}_{a(u,\,v)}
= \underbrace{\int_\Omega g \, v \, d\mathbf{r}}_{L(v)}
$$

The boundary term vanishes because $v = 0$ on $\partial\Omega$.
FEniCS solves $a(u,v) = L(v)$ for all $v$ by expanding $u$ and $v$ in
piecewise-linear basis functions on the mesh triangles.

In [ ]:
# ================================================================
#  PART 1 PARAMETERS  — change these to explore the problem
# ================================================================

# Mesh resolution: number of triangular elements along each edge
nx = 32    # ACTION: try 8, 16, 32, 64
ny = 32

# Normalised load distribution g(x,y)
# FEniCS expressions use C++ syntax: 'x[0]' = x-coord, 'x[1]' = y-coord
#
# Option A — uniform load:
load_str = "1.0"
#
# Option B — Gaussian (central point load), same peak magnitude:
# load_str = "exp(-50*(pow(x[0]-0.5,2) + pow(x[1]-0.5,2)))"
#
# Option C — two symmetric point loads:
# load_str = "exp(-200*(pow(x[0]-0.3,2)+pow(x[1]-0.5,2))) + exp(-200*(pow(x[0]-0.7,2)+pow(x[1]-0.5,2)))"
#
# Option D — linear ramp (asymmetric):
# load_str = "2.0*x[0]"
#
# ACTION: uncomment exactly ONE option above

print(f"Mesh: {nx}x{ny}  |  load: {load_str[:60]}")

In [ ]:
# --- Create mesh and function space ---
# UnitSquareMesh generates a triangulated mesh on [0,1] x [0,1]
mesh = UnitSquareMesh(nx, ny)

# P1 Lagrange elements: piecewise linear shape functions
# Each vertex of the mesh carries one degree of freedom (DOF)
V = FunctionSpace(mesh, "Lagrange", 1)

print(f"Mesh:  {mesh.num_vertices()} vertices, {mesh.num_cells()} triangles")
print(f"DOFs:  {V.dim()}")

# --- Boundary condition: u = 0 everywhere on the boundary ---
bc = DirichletBC(V, Constant(0.0), "on_boundary")

# --- Define the variational problem ---
u_p = TrialFunction(V)   # unknown deflection  (trial function)
v_p = TestFunction(V)    # arbitrary test function

# Load distribution
g = Expression(load_str, degree=2)

# Bilinear form:  a(u,v) = integral of grad(u).grad(v)
a = inner(grad(u_p), grad(v_p)) * dx

# Linear form:    L(v)   = integral of g*v
L_form = g * v_p * dx

# --- Solve: find u such that a(u,v) = L(v) for all v ---
u_sol = Function(V)
solve(a == L_form, u_sol, bc)

print(f"\nSolved!  u_max = {u_sol.vector().max():.6f}")

In [ ]:
# --- Visualise the deflection field ---
# We sample the FEniCS solution on a regular grid for clean matplotlib plots
N_vis = 80
x_pts = np.linspace(0.005, 0.995, N_vis)   # slightly inside the boundary
y_pts = np.linspace(0.005, 0.995, N_vis)

Z = np.zeros((N_vis, N_vis))
for i, yj in enumerate(y_pts):
    for j, xi in enumerate(x_pts):
        Z[i, j] = u_sol(xi, yj)   # FEniCS point evaluation

X, Y = np.meshgrid(x_pts, y_pts)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 2D colour map
im = axes[0].contourf(X, Y, Z, levels=30, cmap='plasma')
plt.colorbar(im, ax=axes[0], label="Deflection u")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_title("Membrane deflection u(x,y)")
axes[0].set_aspect("equal")

# Centreline profile along y = 0.5
profile = [u_sol(xi, 0.5) for xi in x_pts]
axes[1].plot(x_pts, profile, "b-", linewidth=2, label="FEM (y = 0.5)")
axes[1].set_xlabel("x")
axes[1].set_ylabel("u(x, 0.5)")
axes[1].set_title("Centreline deflection profile")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Maximum deflection: {u_sol.vector().max():.6f}")

### Mesh Convergence

A key question in FEM: *how fine does the mesh need to be?*
If you keep refining the mesh and the solution stops changing, it has **converged**.
The cell below solves the uniform-load case on progressively finer meshes and tracks
the maximum deflection.

In [ ]:
print("Running convergence study (uniform load g=1)...")
mesh_sizes   = [4, 8, 16, 32, 64]
max_defl     = []
n_dofs_list  = []

for n in mesh_sizes:
    m   = UnitSquareMesh(n, n)
    V_c = FunctionSpace(m, "Lagrange", 1)
    bc_c = DirichletBC(V_c, Constant(0.0), "on_boundary")
    u_c  = TrialFunction(V_c)
    v_c  = TestFunction(V_c)
    a_c  = inner(grad(u_c), grad(v_c)) * dx
    L_c  = Constant(1.0) * v_c * dx
    u_s  = Function(V_c)
    solve(a_c == L_c, u_s, bc_c)
    max_defl.append(u_s.vector().max())
    n_dofs_list.append(V_c.dim())
    print(f"  n={n:3d}  DOFs={V_c.dim():6d}  u_max={u_s.vector().max():.6f}")

plt.figure(figsize=(7, 4))
plt.semilogx(n_dofs_list, max_defl, "o-", color="steelblue", linewidth=2, markersize=8)
plt.xlabel("Number of DOFs")
plt.ylabel("Maximum deflection")
plt.title("Mesh convergence (uniform load g = 1)")
plt.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()
print(f"\nConverged value:  u_max = {max_defl[-1]:.5f}")

### Task 1: Explore the Membrane Problem

Return to the **Parameters** cell above and try the experiments below.
For each change, re-run the Parameters, Solve, and Visualise cells.

**1.1 — Load distribution**
- Switch to Option B (Gaussian central load). How does the deflection profile change compared
  to the uniform load? Where is the peak deflection?
- For the same peak magnitude, which case produces a larger *maximum* deflection — uniform or Gaussian?

**1.2 — Asymmetric loading**
- Try Option D (linear ramp). Is the solution symmetric? Why or why not?
- Describe the centreline profile: where does the maximum occur?

**1.3 — Two point loads**
- Try Option C. How many deflection peaks do you see? What happens at the centre?

**1.4 — Convergence**
- From the convergence plot: approximately how many DOFs are needed before the solution
  is effectively converged (changes by less than 1%)?

> *Record your observations by adding a markdown cell below each experiment.*

---
## Part 2: Phase Field Modelling — Spinodal Decomposition

### Physical Background

When a binary alloy (e.g. Sn-Pb solder) is rapidly cooled into the **spinodal region** of the
phase diagram, it becomes thermodynamically unstable: any small composition fluctuation
amplifies spontaneously, driving the system towards phase separation *without* needing
nucleation.
This is **spinodal decomposition**.

The **Cahn-Hilliard model** describes this with a scalar order parameter $n(\mathbf{r},t)$
— the local composition:
- $n = -1$: pure component A
- $n = +1$: pure component B

### Free Energy Functional

The total free energy (from Lecture 3) is:

$$
E[n,\nabla n] = \int_\Omega
\underbrace{G(n)}_{\text{bulk energy}}
+ \underbrace{\frac{\kappa}{2}|\nabla n|^2}_{\text{interface energy}}
\, d\mathbf{r}
$$

The **bulk energy** is a double-well potential with minima at $n = \pm 1$:

$$
G(n) = (1 - n^2)^2
$$

Any composition between the two phases is energetically unfavourable in the bulk.
The **gradient term** penalises sharp interfaces; the parameter $\kappa$ sets the interface width.

### Equation of Motion

Mass conservation gives the **Cahn-Hilliard equation**:

$$
\frac{\partial n}{\partial t} = \nabla^2 \tilde{\mu}, \qquad
\tilde{\mu} = \frac{\partial G}{\partial n} - \kappa\nabla^2 n
= \underbrace{-4n(1-n^2)}_{\text{bulk}} - \kappa\nabla^2 n
$$

This is a **fourth-order PDE** in space.
FEniCS handles second-order PDEs naturally, so we split it into two coupled second-order
equations by introducing the chemical potential $\mu$ as an independent variable:

$$
\text{(1)}\quad \frac{n - n_\mathrm{old}}{\Delta t} = \nabla^2 \mu_\theta
\qquad
\text{(2)}\quad \mu = \frac{\partial G}{\partial n} - \kappa\nabla^2 n
$$

where $\mu_\theta = \theta\,\mu + (1-\theta)\,\mu_\mathrm{old}$ is a
Crank-Nicholson time average ($\theta = 0.5$).  
We solve for $n$ and $\mu$ **simultaneously** using a mixed finite element space.

In [ ]:
# ================================================================
#  PART 2 PARAMETERS  — change these to explore phase separation
# ================================================================

# --- Physical parameters ---
kappa  = 0.01   # Interface energy coefficient (controls interface width)
                # ACTION: try 0.001, 0.005, 0.01, 0.05

# --- Initial conditions ---
n_mean = 0.0    # Mean composition  (-1 < n_mean < 1)
                # ACTION: try -0.3, 0.0, 0.3
noise  = 0.05   # Amplitude of initial random fluctuations (keeps it small)

# --- Numerical parameters ---
nx_ch  = 64     # Mesh divisions per side  (larger = more detail, slower)
ny_ch  = 64
dt     = 1.0e-4 # Time step  — reduce if the solver does not converge
                # ACTION: try 5e-5 or 5e-6
theta  = 0.5    # 0.5 = Crank-Nicholson,  1.0 = backward Euler
                # ACTION: try 1.0 to compare stability

# --- Simulation length ---
n_steps    = 1000  # Total time steps
                   # ACTION: increase to 300 to observe late-stage coarsening
save_every = 25    # Save a snapshot every this many steps

print(f"kappa={kappa}  n_mean={n_mean}  noise={noise}")
print(f"Mesh {nx_ch}x{ny_ch}  dt={dt}  theta={theta}")
print(f"{n_steps} steps, snapshot every {save_every}")

In [ ]:
# --- Mesh ---
mesh_ch = UnitSquareMesh(nx_ch, ny_ch)

# --- Mixed function space ---
# We solve for (n, mu) simultaneously.
# Each component uses piecewise-linear (P1) Lagrange elements.
P1 = FiniteElement("Lagrange", mesh_ch.ufl_cell(), 1)
ME = FunctionSpace(mesh_ch, P1 * P1)   # product space: (n, mu)

print(f"Mixed FE space:  {ME.dim()} total DOFs  ({(nx_ch+1)*(ny_ch+1)} per component)")

# Trial and test functions for the mixed space
du   = TrialFunction(ME)       # increment used in Newton linearisation
q, v = TestFunctions(ME)       # test functions for equations (1) and (2)

# Solution functions: current and previous time step
u_new = Function(ME)   # (n, mu) at the new time step
u_old = Function(ME)   # (n, mu) at the previous time step

# Symbolic split — returns UFL sub-expressions for use in the weak form
n,  mu  = split(u_new)
n0, mu0 = split(u_old)

# --- Initial conditions ---
# Start from a nearly uniform mixture (n = n_mean) with small random noise.
# Without noise the perfectly uniform state would never break its symmetry.
class InitialConditions(UserExpression):
    def __init__(self, mean, amplitude, seed=42, **kwargs):
        super().__init__(**kwargs)
        self.mean = mean
        self.amplitude = amplitude
        random.seed(seed)

    def eval(self, values, x):
        values[0] = self.mean + self.amplitude * (2*random.random() - 1)  # n
        values[1] = 0.0                                                     # mu

    def value_shape(self):
        return (2,)   # two-component output

u_init = InitialConditions(mean=n_mean, amplitude=noise, degree=1)
u_new.interpolate(u_init)
u_old.interpolate(u_init)

# Quick sanity check on the initial field
n_func_init, _ = u_new.split(deepcopy=True)
print(f"Initial n range: [{n_func_init.vector().min():.3f}, {n_func_init.vector().max():.3f}]")

In [ ]:
# --- Free energy and chemical potential via UFL automatic differentiation ---

# Mark n as a UFL variable so diff() can differentiate with respect to it
n_var = variable(n)

# Double-well bulk free energy (from lecture):  G(n) = (1 - n^2)^2
G    = (1 - n_var**2)**2
dGdn = diff(G, n_var)       # automatic differentiation: dG/dn = -4n(1-n^2)

# Crank-Nicholson time average of the chemical potential
mu_mid = theta * mu + (1 - theta) * mu0

# --- Weak form (residual F = 0) ---
#
# Equation (1) — mass conservation:
#   integral[ (n-n0)/dt * q ] dx  +  integral[ grad(mu_theta).grad(q) ] dx  = 0
F0 = ((n - n0) / dt) * q * dx  +  dot(grad(mu_mid), grad(q)) * dx

# Equation (2) — chemical potential definition:
#   integral[ mu * v ] dx  -  integral[ dG/dn * v ] dx  -  kappa * integral[ grad(n).grad(v) ] dx  = 0
F1 = mu * v * dx  -  dGdn * v * dx  -  kappa * dot(grad(n), grad(v)) * dx

F = F0 + F1   # combined residual

# --- Jacobian (needed for Newton's method) ---
# UFL computes the directional derivative of F with respect to u_new automatically.
J = derivative(F, u_new, du)

# --- Solver setup ---
# No Dirichlet BCs: zero-flux Neumann conditions are the natural/default BC
problem = NonlinearVariationalProblem(F, u_new, J=J)
solver  = NonlinearVariationalSolver(problem)

prm = solver.parameters["newton_solver"]
prm["linear_solver"]       = "lu"   # direct solver, reliable for up to ~100x100 mesh
prm["relative_tolerance"]  = 1e-6
prm["maximum_iterations"]  = 15

print("Weak form assembled.")
print("Solver configured  (Newton + LU).")

In [ ]:
# --- Helper: extract composition field and free energy at a given time ---
def save_snapshot(u_now, t_now):
    """Append the composition grid and energy values for time t_now."""
    n_f, _ = u_now.split(deepcopy=True)

    # Extract vertex values and reconstruct the regular grid.
    # For UnitSquareMesh(nx, ny), vertices sit at positions (i/nx, j/ny).
    # Sorting by (y, x) recovers the row-major grid.
    coords   = mesh_ch.coordinates()                  # shape: (n_verts, 2)
    vtx_vals = n_f.compute_vertex_values(mesh_ch)     # values at each vertex
    idx      = np.lexsort((coords[:, 0], coords[:, 1]))
    Z        = vtx_vals[idx].reshape(ny_ch + 1, nx_ch + 1)

    # Compute free energy integrals with FEniCS
    E_bulk  = float(assemble((1 - n_f**2)**2 * dx))
    E_iface = float(assemble(0.5 * kappa * dot(grad(n_f), grad(n_f)) * dx))

    snapshots.append(Z.copy())
    snap_times.append(t_now)
    energies.append((t_now, E_bulk, E_iface))


# Initialise storage
snapshots  = []
snap_times = []
energies   = []

save_snapshot(u_new, 0.0)   # record the initial state

# --- Time stepping loop ---
print("Starting Cahn-Hilliard simulation...")
t_sim = 0.0
t0    = timer.time()

for step in range(1, n_steps + 1):

    # Archive the current solution before advancing
    u_old.assign(u_new)

    # Solve the nonlinear system for the next time step
    solver.solve()
    t_sim += dt

    if step % save_every == 0:
        save_snapshot(u_new, t_sim)
        E_tot   = energies[-1][1] + energies[-1][2]
        elapsed = timer.time() - t0
        print(f"  step {step:4d}/{n_steps}   t={t_sim:.2e}   E={E_tot:.4f}   [{elapsed:.0f}s]")

print(f"\nDone. {len(snapshots)} snapshots saved.")

In [ ]:
# --- Visualise composition snapshots ---
n_snaps = len(snapshots)
cols    = min(n_snaps, 5)
rows    = (n_snaps + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows + 0.5),
                         squeeze=False)
axes = axes.flatten()

for i, (Z, t_val) in enumerate(zip(snapshots, snap_times)):
    im = axes[i].imshow(
        Z, origin="lower", vmin=-1, vmax=1,
        cmap="RdBu_r", extent=[0, 1, 0, 1], interpolation="bilinear"
    )
    axes[i].set_title(f"t = {t_val:.1e}", fontsize=10)
    axes[i].set_xlabel("x", fontsize=9)
    if i % cols == 0:
        axes[i].set_ylabel("y", fontsize=9)

for i in range(n_snaps, len(axes)):
    axes[i].set_visible(False)

fig.subplots_adjust(right=0.88, hspace=0.45, wspace=0.3)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cb = fig.colorbar(im, cax=cbar_ax)
cb.set_label("Composition n", fontsize=10)
cb.set_ticks([-1, 0, 1])
cb.set_ticklabels(["-1 (Phase A)", "0", "+1 (Phase B)"])

fig.suptitle(f"Spinodal decomposition  (kappa={kappa}, n_mean={n_mean})",
             fontsize=12, y=1.02)
plt.show()

In [ ]:
# --- Plot free energy evolution ---
times_e  = [e[0] for e in energies]
E_bulk_e = [e[1] for e in energies]
E_iface_e= [e[2] for e in energies]
E_total_e= [b + g_e for b, g_e in zip(E_bulk_e, E_iface_e)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Energy components
axes[0].plot(times_e, E_total_e, "k-o",  label="Total E",          linewidth=2, markersize=5)
axes[0].plot(times_e, E_bulk_e,  "b--s", label="Bulk energy G(n)",  linewidth=2, markersize=5)
axes[0].plot(times_e, E_iface_e, "r--^", label="Interface energy",  linewidth=2, markersize=5)
axes[0].set_xlabel("Time t")
axes[0].set_ylabel("Free energy E")
axes[0].set_title("Free energy components")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Relative decrease
E0  = E_total_e[0]
pct = [(E - E0) / abs(E0) * 100 for E in E_total_e]
axes[1].plot(times_e, pct, "k-o", linewidth=2, markersize=5)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[1].set_xlabel("Time t")
axes[1].set_ylabel("Relative change (%)")
axes[1].set_title("Total free energy decrease")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

drop = (E_total_e[-1] - E_total_e[0]) / abs(E_total_e[0]) * 100
print(f"Free energy decreased by {drop:.1f}% over {n_steps} steps.")
print("The system evolves spontaneously towards lower free energy, as required by thermodynamics.")

### Task 2: Explore Spinodal Decomposition

Return to the **Parameters** cell above and try the experiments below.
Re-run from the Parameters cell each time.

**2.1 — Interface energy $\kappa$**  
Try `kappa = 0.001, 0.005, 0.01, 0.05`.
- How does the characteristic size of the phase-separated domains change with $\kappa$?
  (*Hint: the spinodal wavelength scales as $\lambda \propto \sqrt{\kappa}$.)
- Does a smaller $\kappa$ lead to finer or coarser microstructure?

**2.2 — Initial composition $\bar{n}_0$**  
Try `n_mean = 0.0, 0.2, 0.4`.
- Describe the morphology: do you see an interconnected network, or isolated droplets of
  one phase in a matrix of the other?
- The transition between these morphologies is the **percolation transition**.

**2.3 — Late-stage coarsening**  
Set `n_steps = 1000` (keeping $\kappa = 0.001$, `n_mean = 0.0`).
- Observe how domains **coarsen** (merge) over time — this is **Ostwald ripening**.
- Does the free energy continue to decrease throughout?

**2.4 — Time integration scheme**  
Try `theta = 1.0` (backward Euler) and a larger time step `dt = 5e-5`.
- Is the fully implicit scheme more or less stable than Crank-Nicholson?
- What happens if you try `dt = 1e-4`?

---
## Summary

In this lab you have:

- **Solved the Poisson equation** for membrane deflection using FEM in FEniCS
- **Verified mesh convergence** and understood the computational cost of refinement
- **Simulated spinodal decomposition** with the Cahn-Hilliard phase field model
- **Observed** how $\kappa$ controls microstructure length scale and how off-symmetric
  compositions lead to qualitatively different morphologies
- **Tracked free energy** evolving downhill towards thermodynamic equilibrium

| Model | Equation | Key parameter | Example application |
|---|---|---|---|
| Poisson | $-\nabla^2 u = g$ | Load distribution $g$ | Elasticity, heat, diffusion |
| Cahn-Hilliard | $\partial_t n = \nabla^2(\partial G/\partial n - \kappa\nabla^2 n)$ | Interface energy $\kappa$ | Phase separation, microstructure |

Both are solved with the same tool — the **Finite Element Method** — which converts
continuous PDEs into finite algebraic systems once you provide the weak form.

### Further Reading

- **FEniCS tutorial**: *Solving PDEs in Python* — Langtangen & Logg (free online at [fenicsproject.org](https://fenicsproject.org/pub/tutorial/html/))
- **Original paper**: Cahn & Hilliard, *J. Chem. Phys.* **28**(2), 258 (1958)
- **Phase field review**: Moelans, Blanpain & Wollants, *CALPHAD* **32**(2), 268 (2008)